# DIV2K (10 BPP-diverse) — LoRA vs full_rank RD

Mean over images for each $\lambda$: Bpp vs PSNR of compressed SR vs GT.

| Curve set | Directory | Tag pattern |
|-----------|-----------|-------------|
| Nina direct LoRA | `runs/bitrate_sr_nina_direct_10` | `*_r4_direct` |
| Nina ADMM LoRA | `runs/bitrate_sr_nina_admm_10` | `*_r4_admm` |
| Nina direct full_rank | `runs/bitrate_sr_nina_direct_full_rank_10` | `*_direct_full_rank` |
| Nina ADMM full_rank | `runs/bitrate_sr_nina_admm_full_rank_10` | `*_admm_full_rank` |
| Swin direct LoRA | `runs/bitrate_sr_swin_direct_10` | `*_r4_direct` |
| Swin ADMM LoRA | `runs/bitrate_sr_swin_admm_10` | `*_r4_admm` |
| Swin direct full_rank | `runs/bitrate_sr_swin_direct_full_rank_10` | `*_direct_full_rank` |
| Swin ADMM full_rank | `runs/bitrate_sr_swin_admm_full_rank_10` | `*_admm_full_rank` |

Partial runs still plot from available `metrics.json`.

In [ ]:
from pathlib import Path
import json
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt

PROJECT = Path("/gpfs/gpfs0/timofey.glukhikh/Science_Phan")
if not PROJECT.exists():
    PROJECT = Path("/gpfs/data/gpfs0/timofey.glukhikh/Science_Phan")

RUNS = PROJECT / "runs"
OUT_DIR = RUNS / "rd_compare_lora_vs_full_rank"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# (runs_dir, backbone, method_tag, adapt)
SERIES = {
    "nina_direct_lora": (RUNS / "bitrate_sr_nina_direct_10", "nina", "direct", "lora"),
    "nina_admm_lora": (RUNS / "bitrate_sr_nina_admm_10", "nina", "admm", "lora"),
    "nina_direct_full_rank": (RUNS / "bitrate_sr_nina_direct_full_rank_10", "nina", "direct", "full_rank"),
    "nina_admm_full_rank": (RUNS / "bitrate_sr_nina_admm_full_rank_10", "nina", "admm", "full_rank"),
    "swin_direct_lora": (RUNS / "bitrate_sr_swin_direct_10", "swin", "direct", "lora"),
    "swin_admm_lora": (RUNS / "bitrate_sr_swin_admm_10", "swin", "admm", "lora"),
    "swin_direct_full_rank": (RUNS / "bitrate_sr_swin_direct_full_rank_10", "swin", "direct", "full_rank"),
    "swin_admm_full_rank": (RUNS / "bitrate_sr_swin_admm_full_rank_10", "swin", "admm", "full_rank"),
}


def _glob_pat(backbone: str, method: str, adapt: str) -> str:
    if adapt == "full_rank":
        return f"img*_{backbone}_psnr35_lam*_{method}_full_rank"
    return f"img*_{backbone}_psnr35_lam*_r4_{method}"


def load_mean_rd(root: Path, backbone: str, method: str, adapt: str):
    """Return (lams, mean_bpp, mean_psnr, n_per_lam) sorted by lambda."""
    by_lam: dict[float, list[tuple[float, float]]] = defaultdict(list)
    if not root.exists():
        return [], [], [], []

    pat = _glob_pat(backbone, method, adapt)
    for d in root.glob(pat):
        mf = d / "metrics.json"
        if not mf.exists():
            continue
        m = json.loads(mf.read_text())
        lam, bpp, psnr = m.get("lambda"), m.get("Bpp"), m.get("PSNR_cmpref")
        if lam is None or bpp is None or psnr is None:
            continue
        by_lam[float(lam)].append((float(bpp), float(psnr)))

    lams = sorted(by_lam.keys())
    xs, ys, ns = [], [], []
    for lam in lams:
        pts = by_lam[lam]
        xs.append(float(np.mean([p[0] for p in pts])))
        ys.append(float(np.mean([p[1] for p in pts])))
        ns.append(len(pts))
    return lams, xs, ys, ns


def cache_all():
    data = {}
    for key, (root, backbone, method, adapt) in SERIES.items():
        lams, xs, ys, ns = load_mean_rd(root, backbone, method, adapt)
        data[key] = {"root": root, "lams": lams, "bpp": xs, "psnr": ys, "n": ns}
        n_dirs = len(list(root.glob(_glob_pat(backbone, method, adapt)))) if root.exists() else 0
        print(
            f"{key:26s}  exists={root.exists()}  dirs≈{n_dirs:4d}  "
            f"λ points={len(lams)}  n/λ={ns if ns else '-'}"
        )
    return data


DATA = cache_all()

In [ ]:
STYLES = {
    "nina_direct_lora": ("o-", "C0", "Nina direct LoRA"),
    "nina_admm_lora": ("s--", "C1", "Nina ADMM LoRA"),
    "nina_direct_full_rank": ("o-", "C2", "Nina direct full_rank"),
    "nina_admm_full_rank": ("s--", "C3", "Nina ADMM full_rank"),
    "swin_direct_lora": ("^-", "C4", "Swin direct LoRA"),
    "swin_admm_lora": ("d--", "C5", "Swin ADMM LoRA"),
    "swin_direct_full_rank": ("^-", "C6", "Swin direct full_rank"),
    "swin_admm_full_rank": ("d--", "C7", "Swin ADMM full_rank"),
}


def plot_compare(keys: list[str], title: str, save_name: str):
    fig, ax = plt.subplots(figsize=(7.5, 5))
    any_curve = False
    for key in keys:
        d = DATA[key]
        if not d["lams"]:
            print(f"skip {key}: no metrics yet")
            continue
        any_curve = True
        fmt, color, label = STYLES[key]
        order = np.argsort(d["bpp"])
        xs = [d["bpp"][i] for i in order]
        ys = [d["psnr"][i] for i in order]
        lams = [d["lams"][i] for i in order]
        ax.plot(xs, ys, fmt, color=color, markersize=8, label=f"{label} (n≈{max(d['n'])})")
        for x, y, lam in zip(xs, ys, lams):
            ax.annotate(
                f"λ={lam:g}",
                (x, y),
                textcoords="offset points",
                xytext=(4, 4),
                fontsize=7,
                color=color,
            )
    ax.set_xlabel("Bpp (mean over images)")
    ax.set_ylabel("PSNR vs GT compressed (mean)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    if any_curve:
        ax.legend()
    out = OUT_DIR / save_name
    fig.tight_layout()
    fig.savefig(out, dpi=150, bbox_inches="tight")
    print(f"Saved -> {out}")
    plt.show()
    plt.close(fig)

## 1. Nina direct: LoRA vs full_rank

In [ ]:
plot_compare(
    ["nina_direct_lora", "nina_direct_full_rank"],
    title="Nina — direct: LoRA vs full_rank (10 images)",
    save_name="rd_nina_direct_lora_vs_full_rank.png",
)

## 2. Nina ADMM: LoRA vs full_rank

In [ ]:
plot_compare(
    ["nina_admm_lora", "nina_admm_full_rank"],
    title="Nina — ADMM: LoRA vs full_rank (10 images)",
    save_name="rd_nina_admm_lora_vs_full_rank.png",
)

## 3. Swin direct: LoRA vs full_rank

In [ ]:
plot_compare(
    ["swin_direct_lora", "swin_direct_full_rank"],
    title="Swin — direct: LoRA vs full_rank (10 images)",
    save_name="rd_swin_direct_lora_vs_full_rank.png",
)

## 4. Swin ADMM: LoRA vs full_rank

In [ ]:
plot_compare(
    ["swin_admm_lora", "swin_admm_full_rank"],
    title="Swin — ADMM: LoRA vs full_rank (10 images)",
    save_name="rd_swin_admm_lora_vs_full_rank.png",
)

## 5. Nina: all four (direct/ADMM × LoRA/full_rank)

In [ ]:
plot_compare(
    ["nina_direct_lora", "nina_admm_lora", "nina_direct_full_rank", "nina_admm_full_rank"],
    title="Nina — LoRA vs full_rank × direct/ADMM (10 images)",
    save_name="rd_nina_all_lora_vs_full_rank.png",
)